
# FairWarn-SHS — Residence-Aware Fairness Regularization and Ablation

This notebook compares:

1. **Standard GraphSAGE**: residence fairness weight \(\lambda=0\)
2. **Fairness-aware GraphSAGE**: class-weighted prediction loss plus one residence
   equal-opportunity regularization term

The tested strengths are:

```text
λ = 0.00, 0.01, 0.05, 0.10, 0.20
```

For each of five fixed seeds, the final fairness strength is selected using
**validation data only**. The selection rule chooses the smallest validation
residence equal-opportunity gap among models whose validation AUC-PR is within
0.02 of the best validation AUC-PR for that seed.

The held-out test set is used only after model and λ selection.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib scipy

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import random
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"

SEEDS = [42, 123, 456, 789, 1010]
LAMBDAS = [0.00, 0.01, 0.05, 0.10, 0.20]

UTILITY_TOLERANCE = 0.02
MAX_EPOCHS = 500
PATIENCE = 40

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)


In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask_np = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

node_to_index = {
    node_id: index
    for index, node_id in enumerate(nodes["Node_ID"])
}

edge_pairs = []

for _, row in edges.iterrows():
    source_id = row["Source_Node_ID"]
    target_id = row["Target_Node_ID"]

    if (
        source_id in node_to_index
        and target_id in node_to_index
    ):
        source_index = node_to_index[source_id]
        target_index = node_to_index[target_id]

        edge_pairs.extend([
            (source_index, target_index),
            (target_index, source_index),
        ])

edge_index = torch.tensor(
    edge_pairs,
    dtype=torch.long,
).t().contiguous()

excluded_columns = {
    "Node_ID",
    "Roster_Code",
    "School_Code",
    "Class_Code",
    "Label_Available",
    "TARGET_AtRisk",
}

feature_columns = [
    column
    for column in nodes.columns
    if column not in excluded_columns
]

X_raw = nodes[feature_columns].copy()

numeric_columns = (
    X_raw
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

categorical_columns = [
    column
    for column in X_raw.columns
    if column not in numeric_columns
]

for column in numeric_columns:
    X_raw[column] = X_raw[column].fillna(
        X_raw[column].median()
    )

for column in categorical_columns:
    mode = X_raw[column].mode(dropna=True)

    fill_value = (
        mode.iloc[0]
        if not mode.empty
        else "Missing"
    )

    X_raw[column] = X_raw[column].fillna(
        fill_value
    )

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns,
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
        categorical_columns,
    ),
])

X = preprocessor.fit_transform(
    X_raw
).astype(np.float32)

y = (
    nodes["TARGET_AtRisk"]
    .fillna(-1)
    .astype(int)
    .to_numpy()
)

residence_text = (
    nodes["Q6_Residence"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Boarding = 0; Day = 1; anything else = -1
residence_group = np.full(
    len(nodes),
    -1,
    dtype=np.int64,
)

residence_group[
    residence_text.str.contains("board", regex=False)
] = 0

residence_group[
    residence_text.str.contains("day", regex=False)
] = 1

graph_data = Data(
    x=torch.tensor(
        X,
        dtype=torch.float32,
    ),
    edge_index=edge_index,
    y=torch.tensor(
        y,
        dtype=torch.long,
    ),
    residence=torch.tensor(
        residence_group,
        dtype=torch.long,
    ),
)

print("Nodes:", graph_data.num_nodes)
print("Labelled nodes:", int(labelled_mask_np.sum()))
print("At-risk labelled nodes:", int((y[labelled_mask_np] == 1).sum()))
print("Not-at-risk labelled nodes:", int((y[labelled_mask_np] == 0).sum()))
print("Undirected edges:", edge_index.shape[1] // 2)
print("Boarding nodes:", int((residence_group == 0).sum()))
print("Day nodes:", int((residence_group == 1).sum()))
print("Unmapped residence values:", int((residence_group == -1).sum()))
print("Encoded feature dimension:", graph_data.num_node_features)


In [ ]:

def make_masks(seed):
    labelled_indices = np.where(
        labelled_mask_np
    )[0]

    labelled_y = y[labelled_indices]

    train_validation_indices, test_indices = train_test_split(
        labelled_indices,
        test_size=0.20,
        stratify=labelled_y,
        random_state=seed,
    )

    train_validation_y = y[
        train_validation_indices
    ]

    train_indices, validation_indices = train_test_split(
        train_validation_indices,
        test_size=0.1875,
        stratify=train_validation_y,
        random_state=seed,
    )

    masks = []

    for indices in [
        train_indices,
        validation_indices,
        test_indices,
    ]:
        mask = torch.zeros(
            len(y),
            dtype=torch.bool,
        )
        mask[indices] = True
        masks.append(mask)

    return masks


class FairWarnGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            64,
            aggr="mean",
        )

        self.conv2 = SAGEConv(
            64,
            32,
            aggr="mean",
        )

        self.classifier = torch.nn.Linear(
            32,
            2,
        )

        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = self.conv1(
            x,
            edge_index,
        )
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        x = self.conv2(
            x,
            edge_index,
        )
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        return self.classifier(x)


In [ ]:

def residence_soft_equal_opportunity_gap(
    positive_probability,
    labels,
    residence,
    mask,
):
    positive_mask = (
        mask
        & labels.eq(1)
        & residence.ge(0)
    )

    boarding_positive_mask = (
        positive_mask
        & residence.eq(0)
    )

    day_positive_mask = (
        positive_mask
        & residence.eq(1)
    )

    if (
        boarding_positive_mask.sum() == 0
        or day_positive_mask.sum() == 0
    ):
        return torch.tensor(
            0.0,
            device=positive_probability.device,
        )

    boarding_mean = positive_probability[
        boarding_positive_mask
    ].mean()

    day_mean = positive_probability[
        day_positive_mask
    ].mean()

    return torch.abs(
        boarding_mean - day_mean
    )


def hard_residence_metrics(
    true_labels,
    probabilities,
    predictions,
    residence_values,
):
    rows = []

    group_map = {
        0: "Boarding",
        1: "Day",
    }

    for group_code, group_name in group_map.items():
        group_mask = residence_values == group_code

        group_true = true_labels[group_mask]
        group_probability = probabilities[group_mask]
        group_prediction = predictions[group_mask]

        if len(group_true) == 0:
            continue

        tn, fp, fn, tp = confusion_matrix(
            group_true,
            group_prediction,
            labels=[0, 1],
        ).ravel()

        recall = recall_score(
            group_true,
            group_prediction,
            zero_division=0,
        )

        false_positive_rate = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        )

        rows.append({
            "Residence": group_name,
            "N": len(group_true),
            "AtRisk_N": int(
                np.sum(group_true == 1)
            ),
            "NotAtRisk_N": int(
                np.sum(group_true == 0)
            ),
            "Selection_Rate": float(
                np.mean(group_prediction == 1)
            ),
            "Precision_AtRisk": precision_score(
                group_true,
                group_prediction,
                zero_division=0,
            ),
            "Recall_AtRisk": recall,
            "F1_AtRisk": f1_score(
                group_true,
                group_prediction,
                zero_division=0,
            ),
            "False_Positive_Rate": false_positive_rate,
            "False_Negative_Rate": (
                fn / (fn + tp)
                if (fn + tp) > 0
                else np.nan
            ),
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
        })

    group_table = pd.DataFrame(rows)

    if len(group_table) < 2:
        gaps = {
            "Residence_Demographic_Parity_Gap": np.nan,
            "Residence_Equal_Opportunity_Gap": np.nan,
            "Residence_FPR_Gap": np.nan,
            "Residence_Equalized_Odds_Gap": np.nan,
            "Residence_F1_Gap": np.nan,
        }
    else:
        demographic_parity_gap = (
            group_table["Selection_Rate"].max()
            - group_table["Selection_Rate"].min()
        )

        equal_opportunity_gap = (
            group_table["Recall_AtRisk"].max()
            - group_table["Recall_AtRisk"].min()
        )

        false_positive_rate_gap = (
            group_table["False_Positive_Rate"].max()
            - group_table["False_Positive_Rate"].min()
        )

        f1_gap = (
            group_table["F1_AtRisk"].max()
            - group_table["F1_AtRisk"].min()
        )

        gaps = {
            "Residence_Demographic_Parity_Gap": demographic_parity_gap,
            "Residence_Equal_Opportunity_Gap": equal_opportunity_gap,
            "Residence_FPR_Gap": false_positive_rate_gap,
            "Residence_Equalized_Odds_Gap": max(
                equal_opportunity_gap,
                false_positive_rate_gap,
            ),
            "Residence_F1_Gap": f1_gap,
        }

    return group_table, gaps


def overall_metrics(
    true_labels,
    probabilities,
    predictions,
):
    return {
        "AUC_ROC": roc_auc_score(
            true_labels,
            probabilities,
        ),
        "AUC_PR": average_precision_score(
            true_labels,
            probabilities,
        ),
        "Precision_AtRisk": precision_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "Recall_AtRisk": recall_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "F1_AtRisk": f1_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            true_labels,
            predictions,
        ),
        "Accuracy": accuracy_score(
            true_labels,
            predictions,
        ),
        "Brier_Score": brier_score_loss(
            true_labels,
            probabilities,
        ),
    }


In [ ]:

def train_lambda_for_seed(
    seed,
    fairness_lambda,
):
    set_seed(seed)

    train_mask, validation_mask, test_mask = make_masks(
        seed
    )

    graph = graph_data.clone()
    graph.train_mask = train_mask
    graph.validation_mask = validation_mask
    graph.test_mask = test_mask
    graph = graph.to(device)

    model = FairWarnGraphSAGE(
        graph.num_node_features
    ).to(device)

    train_labels = graph.y[
        graph.train_mask
    ]

    class_counts = torch.bincount(
        train_labels,
        minlength=2,
    ).float()

    class_weights = (
        class_counts.sum()
        / (2.0 * class_counts.clamp_min(1.0))
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4,
    )

    best_state = None
    best_epoch = 0
    best_validation_auc_pr = -np.inf
    best_validation_eo_gap = np.inf
    best_validation_objective = -np.inf
    wait = 0

    training_history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()
        optimizer.zero_grad()

        logits = model(
            graph.x,
            graph.edge_index,
        )

        positive_probability = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        prediction_loss = F.cross_entropy(
            logits[graph.train_mask],
            graph.y[graph.train_mask],
            weight=class_weights,
        )

        fairness_loss = residence_soft_equal_opportunity_gap(
            positive_probability,
            graph.y,
            graph.residence,
            graph.train_mask,
        )

        total_loss = (
            prediction_loss
            + fairness_lambda * fairness_loss
        )

        total_loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():
            validation_logits = model(
                graph.x,
                graph.edge_index,
            )

            validation_probability_all = torch.softmax(
                validation_logits,
                dim=1,
            )[:, 1]

            validation_predictions_all = torch.argmax(
                validation_logits,
                dim=1,
            )

            validation_true = (
                graph.y[graph.validation_mask]
                .cpu()
                .numpy()
            )

            validation_probability = (
                validation_probability_all[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_prediction = (
                validation_predictions_all[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_residence = (
                graph.residence[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_auc_pr = average_precision_score(
                validation_true,
                validation_probability,
            )

            _, validation_gaps = hard_residence_metrics(
                validation_true,
                validation_probability,
                validation_prediction,
                validation_residence,
            )

            validation_eo_gap = validation_gaps[
                "Residence_Equal_Opportunity_Gap"
            ]

            if np.isnan(validation_eo_gap):
                validation_eo_gap = 1.0

            validation_objective = (
                validation_auc_pr
                - 0.10 * validation_eo_gap
            )

        training_history.append({
            "Seed": seed,
            "Lambda": fairness_lambda,
            "Epoch": epoch,
            "Prediction_Loss": float(
                prediction_loss.item()
            ),
            "Fairness_Loss": float(
                fairness_loss.item()
            ),
            "Total_Loss": float(
                total_loss.item()
            ),
            "Validation_AUC_PR": float(
                validation_auc_pr
            ),
            "Validation_EO_Gap": float(
                validation_eo_gap
            ),
            "Validation_Objective": float(
                validation_objective
            ),
        })

        if (
            validation_objective
            > best_validation_objective + 1e-6
        ):
            best_validation_objective = validation_objective
            best_validation_auc_pr = validation_auc_pr
            best_validation_eo_gap = validation_eo_gap
            best_epoch = epoch
            best_state = deepcopy(
                model.state_dict()
            )
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    model.load_state_dict(
        best_state
    )
    model.eval()

    with torch.no_grad():
        logits = model(
            graph.x,
            graph.edge_index,
        )

        probabilities_all = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        predictions_all = torch.argmax(
            logits,
            dim=1,
        )

    test_indices = (
        torch.where(graph.test_mask)[0]
        .cpu()
        .numpy()
    )

    test_true = (
        graph.y[graph.test_mask]
        .cpu()
        .numpy()
    )

    test_probability = (
        probabilities_all[graph.test_mask]
        .cpu()
        .numpy()
    )

    test_prediction = (
        predictions_all[graph.test_mask]
        .cpu()
        .numpy()
    )

    test_residence = (
        graph.residence[graph.test_mask]
        .cpu()
        .numpy()
    )

    overall = overall_metrics(
        test_true,
        test_probability,
        test_prediction,
    )

    residence_table, fairness_gaps = hard_residence_metrics(
        test_true,
        test_probability,
        test_prediction,
        test_residence,
    )

    result_row = {
        "Seed": seed,
        "Lambda": fairness_lambda,
        "Best_Epoch": best_epoch,
        "Validation_AUC_PR": best_validation_auc_pr,
        "Validation_EO_Gap": best_validation_eo_gap,
        "Validation_Objective": best_validation_objective,
        **overall,
        **fairness_gaps,
    }

    prediction_table = pd.DataFrame({
        "Seed": seed,
        "Lambda": fairness_lambda,
        "Node_Index": test_indices,
        "Node_ID": (
            nodes.iloc[test_indices]["Node_ID"]
            .to_numpy()
        ),
        "True_Label": test_true,
        "Predicted_Label": test_prediction,
        "AtRisk_Probability": test_probability,
        "Residence_Code": test_residence,
        "Residence": np.where(
            test_residence == 0,
            "Boarding",
            np.where(
                test_residence == 1,
                "Day",
                "Unmapped",
            ),
        ),
    })

    residence_table["Seed"] = seed
    residence_table["Lambda"] = fairness_lambda

    history_table = pd.DataFrame(
        training_history
    )

    return {
        "state_dict": deepcopy(best_state),
        "result": result_row,
        "predictions": prediction_table,
        "residence_metrics": residence_table,
        "history": history_table,
    }



## Run the λ ablation

This cell trains 25 models:

```text
5 seeds × 5 λ values = 25 runs
```

The same train, validation and test split is used across λ values within each seed.


In [ ]:

run_store = {}
ablation_rows = []
ablation_predictions = []
ablation_residence_metrics = []
ablation_histories = []

for seed in SEEDS:
    run_store[seed] = {}

    for fairness_lambda in LAMBDAS:
        print(
            f"Training seed={seed}, "
            f"lambda={fairness_lambda:.2f}"
        )

        run_output = train_lambda_for_seed(
            seed,
            fairness_lambda,
        )

        run_store[seed][fairness_lambda] = run_output

        ablation_rows.append(
            run_output["result"]
        )

        ablation_predictions.append(
            run_output["predictions"]
        )

        ablation_residence_metrics.append(
            run_output["residence_metrics"]
        )

        ablation_histories.append(
            run_output["history"]
        )

        result = run_output["result"]

        print(
            f"  Test AUC-PR={result['AUC_PR']:.4f} | "
            f"Recall={result['Recall_AtRisk']:.4f} | "
            f"EO gap={result['Residence_Equal_Opportunity_Gap']:.4f}"
        )

ablation_metrics_df = pd.DataFrame(
    ablation_rows
)

ablation_predictions_df = pd.concat(
    ablation_predictions,
    ignore_index=True,
)

ablation_residence_metrics_df = pd.concat(
    ablation_residence_metrics,
    ignore_index=True,
)

ablation_history_df = pd.concat(
    ablation_histories,
    ignore_index=True,
)


In [ ]:

# Validation-only λ selection for every seed.
selection_rows = []

for seed, seed_group in ablation_metrics_df.groupby(
    "Seed"
):
    best_validation_auc_pr = seed_group[
        "Validation_AUC_PR"
    ].max()

    eligible = seed_group[
        seed_group["Validation_AUC_PR"]
        >= (
            best_validation_auc_pr
            - UTILITY_TOLERANCE
        )
    ].copy()

    selected = eligible.sort_values(
        [
            "Validation_EO_Gap",
            "Lambda",
        ],
        ascending=[
            True,
            True,
        ],
    ).iloc[0]

    selection_rows.append({
        "Seed": seed,
        "Best_Validation_AUC_PR": best_validation_auc_pr,
        "Utility_Tolerance": UTILITY_TOLERANCE,
        "Eligible_Lambda_Count": len(eligible),
        "Selected_Lambda": selected["Lambda"],
        "Selected_Validation_AUC_PR": selected["Validation_AUC_PR"],
        "Selected_Validation_EO_Gap": selected["Validation_EO_Gap"],
    })

lambda_selection_df = pd.DataFrame(
    selection_rows
)

lambda_selection_df


In [ ]:

# Build the paired standard-versus-selected test comparison.
comparison_rows = []
comparison_predictions = []
comparison_residence_metrics = []

for _, selection in lambda_selection_df.iterrows():
    seed = int(selection["Seed"])
    selected_lambda = float(
        selection["Selected_Lambda"]
    )

    standard_run = run_store[seed][0.00]
    selected_run = run_store[seed][selected_lambda]

    standard_row = {
        **standard_run["result"],
        "Model": "Standard GraphSAGE",
        "Selected_Lambda": 0.00,
    }

    selected_row = {
        **selected_run["result"],
        "Model": "FairWarn-SHS",
        "Selected_Lambda": selected_lambda,
    }

    comparison_rows.extend([
        standard_row,
        selected_row,
    ])

    standard_predictions = (
        standard_run["predictions"].copy()
    )
    standard_predictions["Model"] = (
        "Standard GraphSAGE"
    )

    selected_predictions = (
        selected_run["predictions"].copy()
    )
    selected_predictions["Model"] = (
        "FairWarn-SHS"
    )

    comparison_predictions.extend([
        standard_predictions,
        selected_predictions,
    ])

    standard_group_metrics = (
        standard_run["residence_metrics"].copy()
    )
    standard_group_metrics["Model"] = (
        "Standard GraphSAGE"
    )

    selected_group_metrics = (
        selected_run["residence_metrics"].copy()
    )
    selected_group_metrics["Model"] = (
        "FairWarn-SHS"
    )

    comparison_residence_metrics.extend([
        standard_group_metrics,
        selected_group_metrics,
    ])

comparison_seed_metrics_df = pd.DataFrame(
    comparison_rows
)

comparison_predictions_df = pd.concat(
    comparison_predictions,
    ignore_index=True,
)

comparison_residence_metrics_df = pd.concat(
    comparison_residence_metrics,
    ignore_index=True,
)

comparison_seed_metrics_df[
    [
        "Model",
        "Seed",
        "Selected_Lambda",
        "AUC_PR",
        "Recall_AtRisk",
        "F1_AtRisk",
        "Residence_Equal_Opportunity_Gap",
        "Residence_Equalized_Odds_Gap",
    ]
].sort_values(
    ["Seed", "Model"]
)


In [ ]:

summary_metrics = [
    "AUC_ROC",
    "AUC_PR",
    "Precision_AtRisk",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Weighted_F1",
    "Balanced_Accuracy",
    "Accuracy",
    "Brier_Score",
    "Residence_Demographic_Parity_Gap",
    "Residence_Equal_Opportunity_Gap",
    "Residence_FPR_Gap",
    "Residence_Equalized_Odds_Gap",
    "Residence_F1_Gap",
]

comparison_summary_rows = []

for model_name, group in comparison_seed_metrics_df.groupby(
    "Model"
):
    row = {
        "Model": model_name,
        "Seeds": group["Seed"].nunique(),
    }

    for metric in summary_metrics:
        row[f"{metric}_Mean"] = group[
            metric
        ].mean()

        row[f"{metric}_SD"] = group[
            metric
        ].std(ddof=1)

    comparison_summary_rows.append(row)

comparison_summary_df = pd.DataFrame(
    comparison_summary_rows
)

comparison_summary_df


In [ ]:

# Fixed-λ mean ± SD table across five seeds.
lambda_summary_rows = []

for fairness_lambda, group in ablation_metrics_df.groupby(
    "Lambda"
):
    row = {
        "Lambda": fairness_lambda,
        "Seeds": group["Seed"].nunique(),
    }

    for metric in summary_metrics:
        row[f"{metric}_Mean"] = group[
            metric
        ].mean()

        row[f"{metric}_SD"] = group[
            metric
        ].std(ddof=1)

    lambda_summary_rows.append(row)

lambda_summary_df = pd.DataFrame(
    lambda_summary_rows
).sort_values("Lambda")

lambda_summary_df[
    [
        "Lambda",
        "AUC_PR_Mean",
        "AUC_PR_SD",
        "Recall_AtRisk_Mean",
        "F1_AtRisk_Mean",
        "Residence_Equal_Opportunity_Gap_Mean",
        "Residence_Equalized_Odds_Gap_Mean",
    ]
]


In [ ]:

# Paired Wilcoxon signed-rank tests across the five seeds.
# With only five pairs, p-values must be interpreted cautiously.
test_metrics = [
    "AUC_PR",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Balanced_Accuracy",
    "Residence_Equal_Opportunity_Gap",
    "Residence_Equalized_Odds_Gap",
]

statistical_rows = []

for metric in test_metrics:
    standard = (
        comparison_seed_metrics_df[
            comparison_seed_metrics_df["Model"]
            .eq("Standard GraphSAGE")
        ]
        .sort_values("Seed")[metric]
        .to_numpy()
    )

    fair = (
        comparison_seed_metrics_df[
            comparison_seed_metrics_df["Model"]
            .eq("FairWarn-SHS")
        ]
        .sort_values("Seed")[metric]
        .to_numpy()
    )

    difference = fair - standard

    try:
        statistic, p_value = wilcoxon(
            fair,
            standard,
            zero_method="wilcox",
            alternative="two-sided",
        )
    except ValueError:
        statistic = np.nan
        p_value = np.nan

    nonzero = difference[
        difference != 0
    ]

    if len(nonzero) == 0:
        rank_biserial = 0.0
    else:
        ranks = pd.Series(
            np.abs(nonzero)
        ).rank(
            method="average"
        ).to_numpy()

        positive_rank_sum = ranks[
            nonzero > 0
        ].sum()

        negative_rank_sum = ranks[
            nonzero < 0
        ].sum()

        total_rank_sum = (
            positive_rank_sum
            + negative_rank_sum
        )

        rank_biserial = (
            positive_rank_sum
            - negative_rank_sum
        ) / total_rank_sum

    statistical_rows.append({
        "Metric": metric,
        "Standard_Mean": standard.mean(),
        "FairWarn_Mean": fair.mean(),
        "Mean_Difference_FairWarn_Minus_Standard": difference.mean(),
        "Wilcoxon_Statistic": statistic,
        "P_Value_Two_Sided": p_value,
        "Rank_Biserial_Effect_Size": rank_biserial,
        "Pairs": len(difference),
    })

statistical_tests_df = pd.DataFrame(
    statistical_rows
)

statistical_tests_df


In [ ]:

# Utility–fairness trade-off chart.
plot_df = lambda_summary_df.sort_values(
    "Lambda"
)

plt.figure(figsize=(8, 5))

plt.plot(
    plot_df["Residence_Equal_Opportunity_Gap_Mean"],
    plot_df["AUC_PR_Mean"],
    marker="o",
)

for _, row in plot_df.iterrows():
    plt.annotate(
        f"λ={row['Lambda']:.2f}",
        (
            row["Residence_Equal_Opportunity_Gap_Mean"],
            row["AUC_PR_Mean"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel(
    "Residence equal-opportunity gap"
)
plt.ylabel("Test AUC-PR")
plt.title(
    "FairWarn-SHS utility–fairness trade-off"
)
plt.tight_layout()
plt.show()


In [ ]:

# Residence recall comparison.
recall_plot_df = (
    comparison_residence_metrics_df
    .groupby(
        ["Model", "Residence"]
    )["Recall_AtRisk"]
    .agg(["mean", "std"])
    .reset_index()
)

models = recall_plot_df["Model"].unique()
residences = ["Boarding", "Day"]

x = np.arange(
    len(residences)
)
width = 0.35

plt.figure(figsize=(8, 5))

for model_index, model_name in enumerate(models):
    model_data = (
        recall_plot_df[
            recall_plot_df["Model"].eq(
                model_name
            )
        ]
        .set_index("Residence")
        .reindex(residences)
    )

    offset = (
        model_index - 0.5
    ) * width

    plt.bar(
        x + offset,
        model_data["mean"],
        width,
        yerr=model_data["std"],
        capsize=4,
        label=model_name,
    )

plt.xticks(
    x,
    residences,
)
plt.ylabel("At-risk recall")
plt.title(
    "Residence recall before and after fairness regularization"
)
plt.legend()
plt.tight_layout()
plt.show()



## Interpretation safeguards

- A lower fairness gap is desirable, but not at any cost to predictive utility.
- The selected λ is determined from validation results, not test results.
- Five seeds provide paired comparisons, but statistical power remains limited.
- A non-significant result does not prove that the models are identical.
- The fairness term targets residence equal opportunity only; it does not directly
  optimize gender, programme, or school-type fairness.
- Results should be reported as mean ± standard deviation and accompanied by subgroup
  sample sizes.


In [ ]:

ablation_metrics_df.to_csv(
    "fairwarn_lambda_ablation_by_seed.csv",
    index=False,
)

lambda_summary_df.to_csv(
    "fairwarn_lambda_ablation_summary_mean_sd.csv",
    index=False,
)

lambda_selection_df.to_csv(
    "fairwarn_selected_lambda_by_seed.csv",
    index=False,
)

comparison_seed_metrics_df.to_csv(
    "fairwarn_standard_vs_fair_seed_metrics.csv",
    index=False,
)

comparison_summary_df.to_csv(
    "fairwarn_standard_vs_fair_summary_mean_sd.csv",
    index=False,
)

comparison_residence_metrics_df.to_csv(
    "fairwarn_residence_group_metrics.csv",
    index=False,
)

statistical_tests_df.to_csv(
    "fairwarn_paired_statistical_tests.csv",
    index=False,
)

comparison_predictions_df.to_csv(
    "fairwarn_standard_vs_fair_predictions.csv",
    index=False,
)

ablation_history_df.to_csv(
    "fairwarn_training_history.csv",
    index=False,
)

files.download(
    "fairwarn_lambda_ablation_summary_mean_sd.csv"
)

files.download(
    "fairwarn_selected_lambda_by_seed.csv"
)

files.download(
    "fairwarn_standard_vs_fair_summary_mean_sd.csv"
)

files.download(
    "fairwarn_residence_group_metrics.csv"
)

files.download(
    "fairwarn_paired_statistical_tests.csv"
)

files.download(
    "fairwarn_standard_vs_fair_seed_metrics.csv"
)

files.download(
    "fairwarn_standard_vs_fair_predictions.csv"
)

files.download(
    "fairwarn_lambda_ablation_by_seed.csv"
)

files.download(
    "fairwarn_training_history.csv"
)
